# MuFu Scaffold and UMAP Split Evaluation

This notebook compares MuFu and random forest (RF) regression performance when their training and validation molecules are allocated using scaffolds or UMAP/HDBSCAN clusters.

All four conditions use the **same UMAP-defined test set**. The hypothesis is whether changing the training/validation allocation improves prediction on that fixed test set. The evaluation uses the full combined dataset, with five training seeds per condition.

The recorded results show higher test RMSE and lower test R² with UMAP allocation for both models. The sections below walk through the experiment, results, and their interpretation.

## 1. Setup

Import the libraries used to read the results and draw the comparison plots. Locate the `ai` project directory so the notebook can run from either the repository root or the notebook folder.

The five result CSVs are preserved in `artifacts/001/results/`. They were copied only from `ai/outputs/split_methods`, so the report stays tied to the recorded experiment if that output folder changes.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
AI_ROOT = next(
    candidate
    for parent in (cwd, *cwd.parents)
    for candidate in (parent, parent / "ai")
    if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir()
)
ARTIFACTS = AI_ROOT / "notebooks" / "artifacts" / "001"
RESULTS_DIR = ARTIFACTS / "results"

print(f"Saved results: {RESULTS_DIR}")

## 2. Experiment Design

UMAP/HDBSCAN defines an approximately 80%/10%/10% train/validation/test partition using split seed 42. Its test molecules are then removed before the scaffold splitter divides the remaining molecules into training and validation at an 8:1 ratio. Both models are evaluated under both allocations.

| Setting | MuFu | Random forest |
| --- | --- | --- |
| Molecular representation | Frozen MoLFormer, ECFP4 and GINE graph encoders | ECFP4, 2,048 bits |
| Training seeds | 0, 1, 2, 3, 4 | 0, 1, 2, 3, 4 |
| Training/model selection | Up to 50 epochs; validation early stopping; task-family checkpoints | 500 trees per endpoint; validation selection of `max_features` |
| Settings across allocations | Same MuFu parameters | Tune separately per endpoint/allocation at seed 0, then fix across training seeds |

MuFu uses learning rate 0.001, weight decay 0.0001, batch size 64, patience 10, and fingerprint dropout 0.6. RF selected `max_features=0.1` for both endpoints under both allocations. The complete recorded parameters are in [`artifacts/001/configs.json`](artifacts/001/configs.json).

The clustering uses 30 neighbors, five UMAP dimensions, `min_dist=0`, and Jaccard distance on binary fingerprints. HDBSCAN uses `min_cluster_size=10`, `min_samples=10`, and `cluster_selection_method="leaf"`. Non-noise clusters remain whole; noise molecules are assigned individually.

These are **five training seeds on one fixed split**, rather than five independently generated partitions.

In [ ]:
results = pd.read_csv(RESULTS_DIR / "sweep_results.csv")
assignments = pd.read_csv(RESULTS_DIR / "split_assignments.csv", keep_default_na=False)

models = ["mufu_scaffold", "mufu_umap", "rf_scaffold", "rf_umap"]
model_labels = {
    "mufu_scaffold": "MuFu · scaffold",
    "mufu_umap": "MuFu · UMAP",
    "rf_scaffold": "RF · scaffold",
    "rf_umap": "RF · UMAP",
}
endpoints = ["pIC50_BACE1", "pIC50_EGFR"]
split_order = ["train", "val", "test"]

regression = results.loc[
    results["model"].isin(models)
    & results["endpoint"].isin(endpoints)
    & results["metric"].isin(["rmse", "r2"])
].copy()
summary = regression.groupby(["model", "split", "endpoint", "metric"], as_index=False).agg(
    mean=("value", "mean"), sample_std=("value", "std"), n_seeds=("value", "count")
)

print(f"Molecules: {len(assignments):,}")
print("Training seeds:", sorted(regression["seed"].unique().tolist()))
print("Conditions:", ", ".join(model_labels.values()))

## 3. Check the Split Allocations

The experiment contains **27,011 molecules**, with the following fixed allocation sizes:

| Allocation | Train | Validation | Shared test |
| --- | ---: | ---: | ---: |
| Scaffold | 21,644 | 2,669 | 2,698 |
| UMAP | 21,617 | 2,696 | 2,698 |

The cross-tabulation below shows where the same molecules were placed by each method. Scaffold training and validation have **zero molecule overlap** with the UMAP test set.

Molecule separation does not imply scaffold separation. Of the shared test molecules, **1,481** share a scaffold with scaffold-training molecules, compared with **1,452** under UMAP training. This experiment does not enforce a scaffold-disjoint test set.

In [ ]:
allocation_counts = pd.DataFrame({
    method: assignments[f"{method}_split"].value_counts().reindex(split_order, fill_value=0)
    for method in ["scaffold", "umap"]
}).T
display(allocation_counts)

allocation_overlap = pd.crosstab(
    assignments["scaffold_split"], assignments["umap_split"]
).reindex(index=split_order, columns=split_order, fill_value=0)
display(allocation_overlap)

test_mask = assignments["umap_split"].eq("test")
for split in ["train", "val"]:
    overlap = (assignments["scaffold_split"].eq(split) & test_mask).sum()
    print(f"Scaffold {split} molecules in UMAP test: {overlap}")

display(pd.read_csv(RESULTS_DIR / "scaffold_overlap.csv"))

## 4. Shared Test Performance

Compare BACE1 and EGFR regression on the same test molecules. Each condition is scored on **810 labeled BACE1** and **1,035 labeled EGFR** test molecules.

RMSE measures prediction error in pIC50 units, so lower is better. R² measures explained variation, so higher is better. Values below are **mean ± sample standard deviation over five training seeds**, using validation-selected models.

| Condition | BACE1 RMSE ↓ | BACE1 R² ↑ | EGFR RMSE ↓ | EGFR R² ↑ |
| --- | --- | --- | --- | --- |
| MuFu · scaffold | 0.742 ± 0.004 | 0.658 ± 0.004 | 0.814 ± 0.008 | 0.604 ± 0.008 |
| MuFu · UMAP | 0.800 ± 0.013 | 0.603 ± 0.013 | 0.919 ± 0.020 | 0.496 ± 0.021 |
| RF · scaffold | 0.732 ± 0.001 | 0.668 ± 0.001 | 0.802 ± 0.002 | 0.617 ± 0.002 |
| RF · UMAP | 0.794 ± 0.001 | 0.609 ± 0.001 | 0.884 ± 0.001 | 0.534 ± 0.001 |

In [ ]:
def metric_table(split):
    rows = summary.loc[summary["split"].eq(split)].copy()
    rows["mean ± std"] = rows.apply(
        lambda row: f"{row['mean']:.3f} ± {row['sample_std']:.3f}", axis=1
    )
    table = rows.pivot(index="model", columns=["endpoint", "metric"], values="mean ± std")
    columns = pd.MultiIndex.from_product([endpoints, ["rmse", "r2"]])
    return table.reindex(index=models, columns=columns).rename(index=model_labels)

display(metric_table("test"))

## 6. Compare Scaffold and UMAP Within Each Model

Pair results by training seed and endpoint, then subtract scaffold test RMSE from UMAP test RMSE. A positive difference means that UMAP allocation produced higher test error.

The table reports mean ± sample standard deviation of the **five paired test differences**. Both models had higher UMAP test RMSE on both endpoints in every seed.

| Model | Endpoint | Paired Δ test RMSE | Relative change | Seeds with higher UMAP test RMSE |
| --- | --- | --- | --- | --- |
| mufu | pIC50_BACE1 | +0.057 ± 0.011 | +7.7% | 5/5 |
| mufu | pIC50_EGFR | +0.105 ± 0.017 | +12.9% | 5/5 |
| rf | pIC50_BACE1 | +0.062 ± 0.002 | +8.5% | 5/5 |
| rf | pIC50_EGFR | +0.082 ± 0.001 | +10.2% | 5/5 |

In [ ]:
test_rmse = regression.loc[
    regression["split"].eq("test") & regression["metric"].eq("rmse")
]
pair_keys = ["seed", "dataset", "endpoint", "data_sha256", "shared_test_sha256", "encoder"]
paired_results = []

for family in ["mufu", "rf"]:
    scaffold = test_rmse.loc[test_rmse["model"].eq(f"{family}_scaffold")]
    umap = test_rmse.loc[test_rmse["model"].eq(f"{family}_umap")]
    paired = umap[pair_keys + ["value"]].merge(
        scaffold[pair_keys + ["value"]], on=pair_keys,
        suffixes=("_umap", "_scaffold"), validate="one_to_one"
    )
    paired["delta_rmse"] = paired["value_umap"] - paired["value_scaffold"]
    paired["model"] = family.upper()
    paired_results.append(paired)

paired_results = pd.concat(paired_results, ignore_index=True)
paired_summary = paired_results.groupby(["model", "endpoint"]).agg(
    mean_delta=("delta_rmse", "mean"), sample_std=("delta_rmse", "std"),
    n_pairs=("delta_rmse", "count"), seeds_with_higher_umap_error=("delta_rmse", lambda x: x.gt(0).sum())
)
display(paired_summary.round(3))

## 7. Compare MuFu With the RF Baseline

RF provides a second model family for checking whether the response to UMAP allocation is specific to MuFu. It is trained on the same allocation as the corresponding MuFu condition, using the available regression labels for each target.

The **paired test RMSE difference is MuFu minus RF**, reported as mean ± sample standard deviation across five training-seed pairs. Positive values indicate higher MuFu error. RF has lower mean test RMSE under both allocations on both endpoints.

| Allocation | Endpoint | Paired Δ test RMSE: MuFu − RF |
| --- | --- | --- |
| scaffold | pIC50_BACE1 | +0.010 ± 0.003 |
| scaffold | pIC50_EGFR | +0.013 ± 0.009 |
| umap | pIC50_BACE1 | +0.005 ± 0.014 |
| umap | pIC50_EGFR | +0.036 ± 0.020 |

In [ ]:
rf_comparisons = []
pair_keys = ["seed", "dataset", "endpoint", "data_sha256", "shared_test_sha256", "split_sha256"]

for allocation in ["scaffold", "umap"]:
    mufu_rows = test_rmse.loc[test_rmse["model"].eq(f"mufu_{allocation}")]
    rf_rows = test_rmse.loc[test_rmse["model"].eq(f"rf_{allocation}")]
    paired = mufu_rows[pair_keys + ["value"]].merge(
        rf_rows[pair_keys + ["value"]], on=pair_keys,
        suffixes=("_mufu", "_rf"), validate="one_to_one"
    )
    paired["delta_rmse"] = paired["value_mufu"] - paired["value_rf"]
    paired["allocation"] = allocation
    rf_comparisons.append(paired)

rf_comparisons = pd.concat(rf_comparisons, ignore_index=True)
display(rf_comparisons.groupby(["allocation", "endpoint"]).agg(
    mean_delta=("delta_rmse", "mean"), sample_std=("delta_rmse", "std"), n_pairs=("delta_rmse", "count")
).round(3))

## 8. Validation Performance

Validation results help explain model selection. MuFu uses validation data for early stopping and checkpoint selection; RF uses it to select `max_features`.

Unlike the shared test comparison, **the validation molecules differ between allocations**. These values are validation mean ± sample standard deviation across five training seeds and should be interpreted with that difference in mind.

| Condition | BACE1 RMSE ↓ | BACE1 R² ↑ | EGFR RMSE ↓ | EGFR R² ↑ |
| --- | --- | --- | --- | --- |
| MuFu · scaffold | 0.728 ± 0.013 | 0.679 ± 0.011 | 0.812 ± 0.012 | 0.524 ± 0.014 |
| MuFu · UMAP | 0.847 ± 0.010 | 0.568 ± 0.010 | 0.907 ± 0.017 | 0.507 ± 0.019 |
| RF · scaffold | 0.745 ± 0.001 | 0.665 ± 0.001 | 0.744 ± 0.003 | 0.600 ± 0.003 |
| RF · UMAP | 0.867 ± 0.001 | 0.548 ± 0.001 | 0.874 ± 0.001 | 0.543 ± 0.001 |

In [ ]:
display(metric_table("val"))

## 9. Source and Label Coverage

Check how many molecules from each dataset source and how many labeled regression examples were retained in each split. Similar counts help rule out a large loss of labeled training data, although they do not establish similar chemistry or response distributions.

Source flags can overlap, so source counts do not necessarily sum to the total molecule count. The saved binary-label audit also contains both positive and negative examples for all 16 binary endpoints in both allocations and all three splits; class presence alone does not establish sufficient coverage.

In [ ]:
source_coverage = pd.read_csv(RESULTS_DIR / "source_coverage.csv")
display(source_coverage.loc[source_coverage["source"].ne("all")]
        .set_index(["method", "source"])[split_order])

label_coverage = pd.read_csv(RESULTS_DIR / "label_coverage.csv")
regression_coverage = label_coverage.loc[label_coverage["endpoint"].isin(endpoints)]
display(regression_coverage.pivot(
    index=["method", "endpoint"], columns="split", values="n_labeled"
).reindex(columns=split_order))

## 10. Conclusion

For this configuration and fixed shared test, UMAP training/validation allocation did not improve MuFu regression performance. MuFu test RMSE increased from **0.742 to 0.800** for BACE1 and from **0.814 to 0.919** for EGFR. Test R² decreased for both endpoints.

RF worsened in the same direction, with test RMSE increasing from **0.732 to 0.794** for BACE1 and from **0.802 to 0.884** for EGFR. This suggests that the allocation effect is shared across the two model families rather than being specific to MuFu.

These results do not establish that UMAP gives a more realistic deployment estimate. The experiment uses one split seed, both conditions share a UMAP-defined test set, and the training and validation molecules change together. Test molecules can share scaffolds with training. UMAP also sees all molecular fingerprints when constructing the split, though it does not use target labels. Training-seed standard deviations do not measure uncertainty across partitions or establish statistical significance.

### Reproduce the experiment

From `ai/`, the equivalent commands are:

```powershell
python scripts/sweep_split_methods.py prepare
python scripts/sweep_split_methods.py all --seeds 0 1 2 3 4 --epochs 50
python scripts/sweep_split_methods.py summary
```

The driver reuses saved assignments and completed seeds. Use a new `--output-dir` when generating another partition; `--force` repeats training on the existing assignments.